In [3]:
import requests
import pandas as pd

/Users/giovannidisalvo/venv/my_venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [10]:
url = 'https://api.binance.com/api/v3/klines'

params = {
    'symbol':'BTCUSDT',
    'interval':'1d',
    'limit': 1000
}

In [ ]:
response = requests.get(url, params=params)
response.raise_for_status() # raise error if download fails
data = response.json()

columns = [
    'open_time',
    'open',
    'high',
    'low',
    'close',
    'volume',
    'close_time',
    'quote_asset_volume',
    'number_of_trades',
    'taker_buy_base_volume',
    'taker_buy_quote_volume',
    'ignore',
]

df = pd.DataFrame(data, columns=columns)

In [5]:
# CLEANING

# put the correct dtype for the features
for feature, feat_type in df.dtypes.items():
    if feat_type == object:
        df[feature] = df[feature].astype(float)

# convert from ms (standard for Binance) to dd-mm-yyyy format
df['open_time'] = pd.to_datetime(df['open_time'], unit='ms').dt.round('s')

# sanity check for the open_time before setting it as index
assert not df['open_time'].duplicated().any()
assert df['open_time'].is_monotonic_increasing

df = df.set_index('open_time')

# remove useless column
df = df.drop(columns=['close_time', 'ignore'])

# be sure there is no NAN
assert not df.isna().any().any()

In [6]:
# VALIDATION

# basic sanity checks
checks = [
    df['low'] <= df['high'],
    df['low'] <= df['open'],
    df['low'] <= df['close'],
    df['open'] <= df['high'],
    df['close'] <= df['high'],
]

for check in checks:
    if not check.all():
        print('error at ', check)
    assert check.all()